# VOICECLONE-QC RVC Bridge

Notebook propre sans WebUI pour entrainer un modele RVC depuis les ZIP crees par VOICECLONE-QC.

Ordre d'execution: lance les cellules une par une de haut en bas.


In [1]:
#@title 1. Configuration VOICECLONE-QC
MODEL_NAME = "Alertes_Stephanie"  #@param {type:"string"}
TARGET_SAMPLE_RATE = "40k"  #@param ["32k", "40k", "48k"]
MODEL_ARCHITECTURE = "v2"  #@param ["v1", "v2"]
PRETRAIN_TYPE = "OV2"  #@param ["original", "OV2", "RIN_E3"]
PITCH_METHOD = "rmvpe"  #@param ["harvest", "crepe", "mangio-crepe", "rmvpe"]
TOTAL_EPOCHS = 200  #@param {type:"integer"}
SAVE_FREQUENCY = 20  #@param {type:"integer"}
BATCH_SIZE = 7  #@param {type:"integer"}

DRIVE_DATASET_DIR = "/content/drive/MyDrive/VOICECLONE_Datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RVC_Output"
ZIP_PATH = f"{DRIVE_DATASET_DIR}/{MODEL_NAME}_dataset_cleaned.zip"
NOW_DIR = "/content/Mangio-RVC-Fork"
EXP_DIR = f"{NOW_DIR}/logs/{MODEL_NAME}"
DATASET_DIR = f"/content/voiceclone_qc/{MODEL_NAME}/dataset"

print("Model:", MODEL_NAME)
print("Dataset ZIP:", ZIP_PATH)
print("Output Drive:", DRIVE_OUTPUT_DIR)


Model: Alertes_Stephanie
Dataset ZIP: /content/drive/MyDrive/VOICECLONE_Datasets/Alertes_Stephanie_dataset_cleaned.zip
Output Drive: /content/drive/MyDrive/RVC_Output


In [2]:
#@title 2. Install Dependencies
import subprocess
import sys

system_packages = [
    "build-essential",
    "python3-dev",
    "ffmpeg",
    "aria2",
]

python_packages = [
    "faiss-cpu",
    "ffmpeg-python",
    "praat-parselmouth",
    "pyworld",
    "numpy",
    "numba",
    "librosa",
    "tensorboardX",
    "tensorboard",
    "onnx",
    "onnxruntime-gpu",
    "torchcrepe",
    "python-dotenv",
    "av",
    "scikit-learn",
]

def run_command(command, label):
    print(f"\nInstalling: {label}", flush=True)
    try:
        subprocess.check_call(command)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            f"\nINSTALLATION FAILED: {label}\n"
            f"Command: {' '.join(command)}\n"
            "Copy the error shown immediately above this message."
        ) from error

print("Updating package list...", flush=True)
run_command(["apt-get", "update", "-qq"], "system package list")

print("Installing system packages...", flush=True)
for package in system_packages:
    run_command(
        ["apt-get", "install", "-qq", "-y", package],
        f"system package {package}",
    )

print("\nUpdating pip tools...", flush=True)
run_command(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    "pip, setuptools and wheel",
)

print("\nInstalling Python packages...", flush=True)
for package in python_packages:
    run_command(
        [sys.executable, "-m", "pip", "install", "--upgrade", package],
        f"Python package {package}",
    )

print("\nInstalling fairseq-fixed...", flush=True)
run_command(
    [sys.executable, "-m", "pip", "install", "fairseq-fixed"],
    "fairseq-fixed",
)

print("\nDependencies ready.", flush=True)

Updating package list...

Installing: system package list
Installing system packages...

Installing: system package build-essential

Installing: system package python3-dev

Installing: system package ffmpeg

Installing: system package aria2

Updating pip tools...

Installing: pip, setuptools and wheel

Installing Python packages...

Installing: Python package faiss-cpu

Installing: Python package ffmpeg-python

Installing: Python package praat-parselmouth

Installing: Python package pyworld

Installing: Python package numpy

Installing: Python package numba

Installing: Python package librosa

Installing: Python package tensorboardX

Installing: Python package tensorboard

Installing: Python package onnx

Installing: Python package onnxruntime-gpu

Installing: Python package torchcrepe

Installing: Python package python-dotenv

Installing: Python package av

Installing: Python package scikit-learn

Installing fairseq-fixed...

Installing: fairseq-fixed

Dependencies ready.


In [3]:
#@title 3. Clone RVC Repository
import os
import shutil
import subprocess

repo_path = "/content/Mangio-RVC-Fork"

os.chdir('/content')
if not os.path.exists(repo_path):
    print("Cloning public Mangio-RVC-Fork...")
    subprocess.check_call([
        'git', 'clone', '--depth=1',
        'https://github.com/Mangio621/Mangio-RVC-Fork.git',
        repo_path
    ])
else:
    print("Repo already exists.")

torchcrepe_target = os.path.join(repo_path, 'torchcrepe')
if not os.path.exists(torchcrepe_target):
    print("Adding torchcrepe...")
    subprocess.check_call([
        'git', 'clone', '--depth=1',
        'https://github.com/maxrmorrison/torchcrepe.git',
        '/content/torchcrepe'
    ])
    shutil.copytree('/content/torchcrepe/torchcrepe', torchcrepe_target, dirs_exist_ok=True)
    shutil.rmtree('/content/torchcrepe', ignore_errors=True)

os.chdir(repo_path)
os.makedirs(os.path.join(repo_path, 'logs'), exist_ok=True)
os.makedirs(os.path.join(repo_path, 'weights'), exist_ok=True)
print('Repository ready:', os.getcwd())


Cloning public Mangio-RVC-Fork...
Adding torchcrepe...
Repository ready: /content/Mangio-RVC-Fork


In [4]:
#@title 4. GPU Check
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Change runtime type to GPU.')

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory // 1024 // 1024, 'MB')

gpus = '-'.join(str(i) for i in range(torch.cuda.device_count()))
print('Using GPU ids:', gpus)


0 Tesla T4 14912 MB
Using GPU ids: 0


In [5]:
#@title 5. Mount Google Drive
import os
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive is already mounted.')

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print('Drive ready:', DRIVE_OUTPUT_DIR)


Mounted at /content/drive
Drive ready: /content/drive/MyDrive/RVC_Output


In [6]:
#@title 6. Download Pretrained Models and RMVPE
import os

os.chdir(NOW_DIR)
os.makedirs(f'{NOW_DIR}/pretrained', exist_ok=True)
os.makedirs(f'{NOW_DIR}/pretrained_v2', exist_ok=True)
os.makedirs(f'{NOW_DIR}/configs', exist_ok=True)

# Minimal set for RVC v2 / OV2 / 40k, plus Hubert and RMVPE.
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kG.pth?download=true -d {NOW_DIR}/pretrained_v2 -o f0G40k_OV2.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kD.pth?download=true -d {NOW_DIR}/pretrained_v2 -o f0D40k_OV2.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Kit-Lemonfoot/RVC_DidntAsk/resolve/main/hubert_base.pt -d {NOW_DIR} -o hubert_base.pt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Kit-Lemonfoot/RVC_DidntAsk/resolve/main/rmvpe.pt -d {NOW_DIR} -o rmvpe.pt
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Kit-Lemonfoot/RVC_DidntAsk/resolve/main/40k.json -d {NOW_DIR}/configs -o 40k.json

print('Pretrained files ready.')



08/11 21:31:55 [ERROR] CUID#10 - Download aborted. URI=https://huggingface.co/ORVC/Ov2Super/resolve/main/f0Ov2Super40kG.pth?download=true
Exception: [AbstractCommand.cc:351] errorCode=22 URI=https://us.gcp.cdn.hf.co/xet-bridge-us/659c1b96abcd340ceb6d8e7a/9a1175cccb1fc477359230184a3917713b58475351a4e7ab9840fe5c1feda1a4?response-content-disposition=attachment%3B+filename*%3DUTF-8%27%27f0Ov2Super40kG.pth%3B+filename%3D%22f0Ov2Super40kG.pth%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1786487515&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU5YzFiOTZhYmNkMzQwY2ViNmQ4ZTdhLzlhMTE3NWNjY2IxZmM0NzczNTkyMzAxODRhMzkxNzcxM2I1ODQ3NTM1MWE0ZTdhYjk4NDBmZTVjMWZlZGExYTRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NjQ4NzUxNX0sIkJ5dGVSYW5nZSI6eyJFeHBlY3RlZEhlYWRlciI6ImJ5dGVzPTE2NDYyNjQzMi0xOTE4ODk0MDcifX19XX0_&Signature=MEUCIDoRRhRrYmUuH6ALdAU

In [7]:
#@title 7. Load VOICECLONE-QC Dataset ZIP
import os
import shutil
import zipfile
from pathlib import Path

zip_path = Path(ZIP_PATH)
dataset_dir = Path(DATASET_DIR)

if not zip_path.exists():
    raise FileNotFoundError(f'ZIP introuvable: {zip_path}')

if dataset_dir.parent.exists():
    shutil.rmtree(dataset_dir.parent)
dataset_dir.parent.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(dataset_dir.parent)

if not dataset_dir.exists():
    raise RuntimeError(f'Le ZIP ne contient pas dataset/: {zip_path}')

for file in list(dataset_dir.iterdir()):
    if file.suffix.lower() not in ['.wav', '.flac', '.mp3', '.ogg', '.m4a']:
        print('Removing non-audio file:', file)
        file.unlink()

wavs = sorted(dataset_dir.glob('*.wav'))
print('Dataset ready:', dataset_dir)
print('WAV files:', len(wavs))
for wav in wavs[:10]:
    print('-', wav.name)


Dataset ready: /content/voiceclone_qc/Alertes_Stephanie/dataset
WAV files: 1
- Alertes_Stephanie_001_0001.wav


In [8]:
#@title 6b. Setup CSVDB
import os
import csv

os.chdir(NOW_DIR)
os.makedirs('csvdb', exist_ok=True)

with open('csvdb/formanting.csv', 'w', newline='') as frmnt:
    csv.writer(frmnt, delimiter=',').writerow([False, 1.0, 1.0])

with open('csvdb/stop.csv', 'w', newline='') as stp:
    csv.writer(stp, delimiter=',').writerow([False])

DoFormant, Quefrency, Timbre = False, 1.0, 1.0
print('CSVDB ready:', os.path.abspath('csvdb/formanting.csv'))


CSVDB ready: /content/Mangio-RVC-Fork/csvdb/formanting.csv


In [9]:
#@title 8. Preprocess Dataset
import os

now_dir = NOW_DIR
experiment_name = MODEL_NAME
target_sample_rate = TARGET_SAMPLE_RATE
model_architecture = MODEL_ARCHITECTURE
pretrain_type = PRETRAIN_TYPE
speaker_id = 0
cpu_threads = max(1, os.cpu_count() or 2)
exp_dir = EXP_DIR

os.chdir(now_dir)
sr = int(target_sample_rate.rstrip('k')) * 1000
os.makedirs(exp_dir, exist_ok=True)

cmd = 'python trainset_preprocess_pipeline_print.py "%s" %s %s "%s" 1' % (
    DATASET_DIR, sr, cpu_threads, exp_dir
)
print(cmd)
!$cmd


python trainset_preprocess_pipeline_print.py "/content/voiceclone_qc/Alertes_Stephanie/dataset" 40000 2 "/content/Mangio-RVC-Fork/logs/Alertes_Stephanie" 1
start preprocess
['trainset_preprocess_pipeline_print.py', '/content/voiceclone_qc/Alertes_Stephanie/dataset', '40000', '2', '/content/Mangio-RVC-Fork/logs/Alertes_Stephanie', '1']
thread:0:   0% 0/1 [00:00<?, ?it/s]
thread:1: 0it [00:00, ?it/s]
thread:0: 100% 1/1 [00:01<00:00,  1.09s/it]
end preprocess


In [10]:
#@title 8b. Python 3.12 / PyTorch Compatibility Patch
from pathlib import Path

sitecustomize = Path(NOW_DIR) / "sitecustomize.py"
sitecustomize.write_text(r'''
import pkgutil
import importlib.machinery

if not hasattr(importlib.machinery.FileFinder, "find_module"):
    def _voiceclone_find_module(self, fullname, path=None):
        spec = self.find_spec(fullname)
        return None if spec is None else spec.loader
    importlib.machinery.FileFinder.find_module = _voiceclone_find_module

if not hasattr(pkgutil, "ImpImporter"):
    class ImpImporter:
        def __init__(self, *args, **kwargs):
            pass
        def find_module(self, fullname, path=None):
            return None
    pkgutil.ImpImporter = ImpImporter

if not hasattr(pkgutil, "ImpLoader"):
    class ImpLoader:
        pass
    pkgutil.ImpLoader = ImpLoader

try:
    import torch
    _voiceclone_original_torch_load = torch.load

    def _voiceclone_torch_load_compat(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _voiceclone_original_torch_load(*args, **kwargs)

    torch.load = _voiceclone_torch_load_compat
except Exception:
    pass
''', encoding="utf-8")

print("Compatibility patch written:", sitecustomize)


Compatibility patch written: /content/Mangio-RVC-Fork/sitecustomize.py


In [11]:
#@title 9. Feature Extraction RMVPE
import os
from pathlib import Path

now_dir = NOW_DIR
experiment_name = MODEL_NAME
model_architecture = MODEL_ARCHITECTURE
cpu_threads = max(1, os.cpu_count() or 2)
pitch_extraction_algorithm = PITCH_METHOD
crepe_hop_length = 128

os.chdir(now_dir)
os.environ["PYTHONPATH"] = now_dir + ":" + os.environ.get("PYTHONPATH", "")

sitecustomize = Path(now_dir) / "sitecustomize.py"
if not sitecustomize.exists():
    sitecustomize.write_text(r'''
import pkgutil

if not hasattr(pkgutil, "ImpImporter"):
    class ImpImporter:
        pass
    pkgutil.ImpImporter = ImpImporter

if not hasattr(pkgutil, "ImpLoader"):
    class ImpLoader:
        pass
    pkgutil.ImpLoader = ImpLoader

try:
    import torch
    _voiceclone_original_torch_load = torch.load

    def _voiceclone_torch_load_compat(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _voiceclone_original_torch_load(*args, **kwargs)

    torch.load = _voiceclone_torch_load_compat
except Exception:
    pass
''', encoding="utf-8")

cmd = 'PYTHONPATH="%s:$PYTHONPATH" python extract_f0_print.py "%s/logs/%s" %s %s %s' % (
    now_dir,
    now_dir,
    experiment_name,
    cpu_threads,
    pitch_extraction_algorithm,
    crepe_hop_length,
)
print(cmd)
!$cmd

gpu_count = len(gpus.split("-")) if "gpus" in globals() and gpus else 1

cmd = 'PYTHONPATH="%s:$PYTHONPATH" python extract_feature_print.py %s %s %s %s "%s/logs/%s" %s' % (
    now_dir,
    "device",
    gpu_count,
    0,
    0,
    now_dir,
    experiment_name,
    model_architecture,
)
print(cmd)
!$cmd


PYTHONPATH="/content/Mangio-RVC-Fork:$PYTHONPATH" python extract_f0_print.py "/content/Mangio-RVC-Fork/logs/Alertes_Stephanie" 2 rmvpe 128
/usr/local/lib/python3.12/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
['extract_f0_print.py', '/content/Mangio-RVC-Fork/logs/Alertes_Stephanie', '2', 'rmvpe', '128']
Using f0 method: rmvpe
thread:0, f0ing, Hop-Length:128:   0% 0/3 [00:00<?, ?it/s]
  0% 0/2 [00:00<?, ?it/s]
thread:1, f0ing, Hop-Length:128:   0% 0/2 [00:00<?, ?it/s]loading rmvpe model
loading rmvpe model

thread:0, f0ing, Hop-Length:128:  33% 1/3 [00:06<00:13,  6.68s/it]
thread:1, f0ing, Hop-Length:128: 100% 2/2 [00:06<00:00,  3.45s/it]
thread:0, f0ing, Hop-Length:128: 100% 3/3 [00:07<00:00,  2.38s/it]
PYTHONPATH="/content/Mangio-

In [12]:
#@title 9b. Matplotlib / NumPy Compatibility Patch
from pathlib import Path

utils_path = Path(NOW_DIR) / "train" / "utils.py"
text = utils_path.read_text(encoding="utf-8")

if "def _voiceclone_canvas_tostring_rgb" not in text:
    marker = "import matplotlib.pyplot as plt\n"
    patch = '''import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg

if not hasattr(FigureCanvasAgg, "tostring_rgb"):
    def _voiceclone_canvas_tostring_rgb(self):
        return self.buffer_rgba().tobytes()
    FigureCanvasAgg.tostring_rgb = _voiceclone_canvas_tostring_rgb
'''
    if marker in text:
        text = text.replace(marker, patch, 1)
    else:
        text = patch + "\n" + text

text = text.replace(
    'np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")',
    'np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)'
)

text = text.replace(
    "np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep='')",
    "np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)"
)

utils_path.write_text(text, encoding="utf-8")
print("Patched:", utils_path)


Patched: /content/Mangio-RVC-Fork/train/utils.py


In [13]:
#@title 9c. RGB Canvas Patch
from pathlib import Path

utils_path = Path(NOW_DIR) / "train" / "utils.py"
text = utils_path.read_text(encoding="utf-8")

text = text.replace(
    "return self.buffer_rgba().tobytes()",
    "import numpy as _np\n        return _np.asarray(self.buffer_rgba())[:, :, :3].tobytes()"
)

utils_path.write_text(text, encoding="utf-8")
print("RGB patch OK:", utils_path)


RGB patch OK: /content/Mangio-RVC-Fork/train/utils.py


In [ ]:
#@title 10. Train RVC Model
import os
import math
from random import shuffle

now_dir = NOW_DIR
experiment_name = MODEL_NAME
target_sample_rate = TARGET_SAMPLE_RATE
model_architecture = MODEL_ARCHITECTURE
pretrain_type = PRETRAIN_TYPE
speaker_id = 0
exp_dir = EXP_DIR

save_frequency = SAVE_FREQUENCY
total_epochs = TOTAL_EPOCHS
batch_size = BATCH_SIZE
save_only_latest_ckpt = True
cache_all_training_sets = False
save_small_final_model = True

os.chdir(now_dir)
pretrained_base = 'pretrained/' if model_architecture == 'v1' else 'pretrained_v2/'
unpt = f'_{pretrain_type}' if pretrain_type != 'original' else ''
pretrainedD = f'{pretrained_base}f0D{target_sample_rate}{unpt}.pth'
pretrainedG = f'{pretrained_base}f0G{target_sample_rate}{unpt}.pth'

log_interval = 1
li_folder = os.path.join(exp_dir, '1_16k_wavs')
if os.path.isdir(li_folder):
    wav_files = [f for f in os.listdir(li_folder) if f.endswith('.wav')]
    if wav_files:
        log_interval = math.ceil(len(wav_files) / batch_size)
        if log_interval > 1:
            log_interval += 1

cmd = 'python train_nsf_sim_cache_sid_load_pretrain.py -e "%s" -sr %s -f0 %s -bs %s -g %s -te %s -se %s %s %s -l %s -c %s -sw %s -v %s -li %s' % (
    experiment_name, target_sample_rate, 1, batch_size, 0, total_epochs, save_frequency,
    '-pg %s' % pretrainedG, '-pd %s' % pretrainedD,
    1 if save_only_latest_ckpt else 0,
    1 if cache_all_training_sets else 0,
    1 if save_small_final_model else 0,
    model_architecture, log_interval
)
print(cmd)

gt_wavs_dir = f'{exp_dir}/0_gt_wavs'
feature_dir = f'{exp_dir}/3_feature256' if model_architecture == 'v1' else f'{exp_dir}/3_feature768'
f0_dir = f'{exp_dir}/2a_f0'
f0nsf_dir = f'{exp_dir}/2b-f0nsf'
names = (
    set([name.split('.')[0] for name in os.listdir(gt_wavs_dir)])
    & set([name.split('.')[0] for name in os.listdir(feature_dir)])
    & set([name.split('.')[0] for name in os.listdir(f0_dir)])
    & set([name.split('.')[0] for name in os.listdir(f0nsf_dir)])
)
opt = []
for name in names:
    opt.append('%s/%s.wav|%s/%s.npy|%s/%s.wav.npy|%s/%s.wav.npy|%s' % (
        gt_wavs_dir, name, feature_dir, name, f0_dir, name, f0nsf_dir, name, speaker_id
    ))
fea_dim = 256 if model_architecture == 'v1' else 768
for _ in range(2):
    opt.append(f'{now_dir}/logs/mute/0_gt_wavs/mute{target_sample_rate}.wav|{now_dir}/logs/mute/3_feature{fea_dim}/mute.npy|{now_dir}/logs/mute/2a_f0/mute.wav.npy|{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|{speaker_id}')
shuffle(opt)
with open(f'{exp_dir}/filelist.txt', 'w') as f:
    f.write('\n'.join(opt))
print('Filelist written:', len(opt), 'items')
!$cmd


python train_nsf_sim_cache_sid_load_pretrain.py -e "Alertes_Stephanie" -sr 40k -f0 1 -bs 7 -g 0 -te 200 -se 20 -pg pretrained_v2/f0G40k_OV2.pth -pd pretrained_v2/f0D40k_OV2.pth -l 1 -c 0 -sw 1 -v v2 -li 1
Filelist written: 7 items
2026-08-11 21:33:00.696818: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:tensorflow:Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:2026-08-11 21:33:02,925:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
DEBUG:jax._src

In [ ]:
#@title 11. Train Index
import os
import sys
import traceback
import numpy as np
import faiss

now_dir = NOW_DIR
experiment_name = MODEL_NAME
model_architecture = MODEL_ARCHITECTURE
exp_dir = EXP_DIR
feature_dir = f'{exp_dir}/3_feature256' if model_architecture == 'v1' else f'{exp_dir}/3_feature768'

if not os.path.exists(feature_dir):
    raise Exception('No features exist. Run Feature Extraction first.')
files = sorted(os.listdir(feature_dir))
if not files:
    raise Exception('No feature files found. Run Feature Extraction first.')

try:
    from sklearn.cluster import MiniBatchKMeans
except Exception:
    MiniBatchKMeans = None

npys = [np.load(f'{feature_dir}/{name}') for name in files]
big_npy = np.concatenate(npys, 0)
np.random.shuffle(big_npy)

if big_npy.shape[0] > 2e5 and MiniBatchKMeans is not None:
    print('KMeans reduction:', big_npy.shape)
    big_npy = MiniBatchKMeans(
        n_clusters=10000, verbose=True, batch_size=256,
        compute_labels=False, init='random'
    ).fit(big_npy).cluster_centers_

np.save(f'{exp_dir}/total_fea.npy', big_npy)
n_ivf = max(1, min(int(16 * np.sqrt(big_npy.shape[0])), max(1, big_npy.shape[0] // 39)))
print('Index shape:', big_npy.shape, 'n_ivf:', n_ivf)

index = faiss.index_factory(256 if model_architecture == 'v1' else 768, f'IVF{n_ivf},Flat')
index_ivf = faiss.extract_index_ivf(index)
index_ivf.nprobe = 1
print('Training index...')
index.train(big_npy)
faiss.write_index(index, f'{exp_dir}/trained_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{experiment_name}_{model_architecture}.index')
print('Adding vectors...')
for i in range(0, big_npy.shape[0], 8192):
    index.add(big_npy[i:i+8192])
index_path = f'{exp_dir}/added_IVF{n_ivf}_Flat_nprobe_{index_ivf.nprobe}_{experiment_name}_{model_architecture}.index'
faiss.write_index(index, index_path)
print('Index ready:', index_path)


In [ ]:
#@title 12. Export to RVC_Output for VOICECLONE-QC Watchdog
import os
import glob
import shutil
from pathlib import Path

output_dir = Path(DRIVE_OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

weight_candidates = [Path(NOW_DIR) / 'weights' / f'{MODEL_NAME}.pth']
weight_candidates += [Path(p) for p in glob.glob(f'{NOW_DIR}/weights/{MODEL_NAME}_*.pth')]
weight_candidates = [p for p in weight_candidates if p.exists()]
if not weight_candidates:
    raise FileNotFoundError('Aucun .pth trouve dans weights/. Training termine?')
weight_path = max(weight_candidates, key=lambda p: p.stat().st_mtime)

index_candidates = [Path(p) for p in glob.glob(f'{EXP_DIR}/added_*.index')]
if not index_candidates:
    raise FileNotFoundError('Aucun .index trouve. Index Training termine?')
index_path = max(index_candidates, key=lambda p: p.stat().st_mtime)

dst_pth = output_dir / f'{MODEL_NAME}.pth'
dst_index = output_dir / f'{MODEL_NAME}.index'
shutil.copy2(weight_path, dst_pth)
shutil.copy2(index_path, dst_index)

print('Exported:')
print(dst_pth)
print(dst_index)


In [ ]:
#@title 13. Auto-disconnect runtime after successful export
from pathlib import Path
from google.colab import runtime
import time

required_files = [
    Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.pth",
    Path(DRIVE_OUTPUT_DIR) / f"{MODEL_NAME}.index",
]

missing = [str(path) for path in required_files if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Export incomplet. Le runtime reste connecte.\nManquant:\n" + "\n".join(missing)
    )

print("Export confirme:")
for path in required_files:
    print(path)

print("Deconnexion du runtime dans 10 secondes...")
time.sleep(10)

runtime.unassign()


## Ensuite

Retourne dans VOICECLONE-QC, section Cloud, et lance **Surveiller le retour modele** avec le nom du comedien.
